<a href="https://colab.research.google.com/github/sukhadaghewari/tropicalcyclone-ri-bench/blob/main/notebooks/07b_run_openai_gpt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# OpenAI models on a case file (text, or text + picture)

Sends every case of one case file to an OpenAI model and saves the answers, one line per case: `{"case_id": ..., "response": ...}`, where `response` is the full API reply. This is how the existing GPT results in `results/` were made, and it works for any case file: RI (development or test) and the storm-identification probe (development or test).

- **The same call as the existing results.** One user message holding the case's `prompt`, through the Chat Completions API. There is no system prompt, no tools (so no web search), and no other settings. Text-only calls are exactly the ones used for the existing GPT results.
- **Reasoning is each model's default.** `reasoning_effort` is not set, as in the existing runs. The defaults differ between models (GPT-5.4 mini's uses no reasoning tokens). Every reply records how many reasoning tokens it used.
- **Pictures:** cases from the `..._text_image` case files have an `image` field; their combined picture (notebook 06) is sent after the prompt text, as image bytes at `detail: "high"`. It is never sent as a file name.
- **Caching:** OpenAI automatically caches the start of a prompt that repeats across calls, at a lower price. The instruction text is the same in every prompt and comes first, with the picture after it, so most of each prompt is cached. The cached token count is in each reply.
- **Each case is its own call**, 16 at a time. Answers are saved as they arrive. Cases already in the results file are skipped, so a re-run only sends what is missing or failed.

Try one case first: it costs a fraction of a cent and shows the answer is parsed correctly.

## 1. Setup

Works out the repo folder and makes `src/` importable, as in the other notebooks. In Colab it first clones this repo. It also installs the `openai` package if it is missing.

In [ ]:
# Setup: in Colab, clone the repo and move into it. Then put src/ on the import path.
import os, subprocess, sys
from pathlib import Path

REPO = "tropicalcyclone-ri-bench"
if "google.colab" in sys.modules:
    try:
        os.getcwd()
    except FileNotFoundError:              # the folder we were in was deleted (e.g. an old clone): start again
        os.chdir("/content")
if "google.colab" in sys.modules and os.path.basename(os.getcwd()) != REPO:
    if not os.path.exists(REPO):
        subprocess.run(["git", "clone", f"https://github.com/sukhadaghewari/{REPO}.git"], check=True)
    os.chdir(REPO)
if "google.colab" in sys.modules:
    # A clone from an earlier run survives "Restart session": bring it up to date with GitHub.
    if subprocess.run(["git", "pull", "--ff-only"]).returncode != 0:
        print("WARNING: could not update the repo with git pull; using the files as they are.")

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()   # repo root, locally or in Colab
sys.path.insert(0, str(ROOT / "src"))
try:
    import openai  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openai"], check=True)

## 2. Settings

**Case file:** any file in `model_inputs/` (the default is the RI test set). For RI with pictures, use `model_inputs/ri_prediction/<dev or test>/..._text_image.jsonl.gz`.

**Model:** `gpt-5-mini`, `gpt-5.4-mini`, `gpt-5.6-luna` or `gpt-6-luna`, the model names used for the existing results.

**Where results go:** `results/<task>/<dev or test>/openai/`, named like the existing results, for example `results/ri_prediction/test/gpt-5-mini_test_2017-2025_50ri_50nori_text_only.jsonl.gz` (committed), with a working `.jsonl` next to it while the run is in progress (not in git). In Colab, set `USE_DRIVE = True` to save them to Google Drive instead (Colab files vanish when the runtime stops), and copy them into `results/` before committing. With `USE_DRIVE = True` the pictures are also read from Drive.

**API key:** the Colab secret `OPENAI_API_KEY` if there is one, otherwise the `OPENAI_API_KEY` environment variable, otherwise you are asked to paste it (it is never saved).

In [ ]:
import base64
import gzip
import concurrent.futures
import json
import os
import time
from datetime import datetime, timezone
from getpass import getpass

import openai
from openai import OpenAI
from tqdm.auto import tqdm
from prompts import parse_identify_response, parse_response, read_records, response_text

# ---- what to send -------------------------------------------------------------------
CASE_FILE = ROOT / "model_inputs" / "ri_prediction" / "test" / "test_2017-2025_50ri_50nori_text_only.jsonl.gz"
LIMIT = None               # e.g. 5 to send only the first 5 cases

# ---- model ----------------------------------------------------------------------------
MODEL = "gpt-5-mini"       # gpt-5-mini | gpt-5.4-mini | gpt-5.6-luna | gpt-6-luna
IMAGE_DETAIL = "high"
WORKERS = 16               # calls at the same time
RETRIES = 5                # attempts per case for rate limits and server errors

# ---- where things are -----------------------------------------------------------------
USE_DRIVE = False          # True: save results to, and read pictures from, Google Drive (Colab)
DRIVE_DIR = "wacv2027_output"   # folder inside MyDrive, as in notebooks 04 and 06

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    images_root = Path("/content/drive/MyDrive") / DRIVE_DIR
    results_root = images_root / "results"
else:
    images_root = ROOT / "model_inputs"
    results_root = ROOT / "results"

# Results file: named like the existing results for this kind of case file.
stem = CASE_FILE.name.removesuffix(".jsonl.gz")
task = CASE_FILE.parent.parent.name                   # model_inputs/<task>/<dev or test>/<case file>
if task == "storm_identification":
    name = f"{MODEL}_detailed_results_{stem}"
    parse = parse_identify_response
else:                                                 # ri_prediction
    name = f"{MODEL}_{stem}"
    parse = parse_response
cases = read_records(CASE_FILE)
results_dir = results_root / task / cases[0]["split"] / "openai"    # e.g. results/storm_identification/dev/openai/
results_path = results_dir / f"{name}.jsonl"                     # working file, appended as answers arrive (not in git)
results_gz = results_dir / f"{name}.jsonl.gz"                    # compressed copy for git and notebooks 08-09
settings_path = results_dir / f"{name}_settings.json"
results_dir.mkdir(parents=True, exist_ok=True)
if results_gz.exists():                                         # continue from the committed answers (e.g. a fresh clone)
    committed = gzip.decompress(results_gz.read_bytes())
    if not results_path.exists() or committed.count(b"\n") > results_path.read_bytes().count(b"\n"):
        results_path.write_bytes(committed)                     # the committed copy has more answers than the working file

if LIMIT:
    cases = cases[:LIMIT]
WITH_IMAGES = "image" in cases[0]                     # the image case files name each case's picture

# ---- API key ----------------------------------------------------------------------------
if not os.environ.get("OPENAI_API_KEY"):
    try:
        from google.colab import userdata
        os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    except Exception:
        os.environ["OPENAI_API_KEY"] = getpass("OpenAI API key: ")
client = OpenAI()

print(f"{len(cases)} cases from {CASE_FILE.name}{' (with pictures)' if WITH_IMAGES else ''}")
print(f"Results: {results_path}")

## 3. The call

`build_messages` makes the message for one case. Text only: exactly the case's prompt as a plain string, as in the existing runs. A case with a picture: the prompt text first (so the shared instruction text is cached), then the picture.

`run_prompt` sends it. It retries rate-limit, connection and server errors, waiting longer each time. Any other error (a bad request) is not retried: the case is reported and left out, to be resent later.

In [ ]:
def image_part(path):
    """A picture as image bytes (a data URL), so the file name never reaches the model."""
    data = base64.standard_b64encode(Path(path).read_bytes()).decode()
    return {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{data}", "detail": IMAGE_DETAIL}}


def build_messages(case):
    if not WITH_IMAGES:
        return [{"role": "user", "content": case["prompt"]}]
    picture = images_root / case["image"]
    if not picture.exists():
        raise FileNotFoundError(f"no combined picture for {case['case_id']}: run 06_combine_satellite_images.ipynb")
    return [{"role": "user", "content": [{"type": "text", "text": case["prompt"]}, image_part(picture)]}]


def run_prompt(case):
    """One case, one independent call. Returns the reply, or raises after the last failed attempt."""
    for attempt in range(RETRIES):
        try:
            return client.chat.completions.create(model=MODEL, messages=build_messages(case))
        except (openai.RateLimitError, openai.APIConnectionError, openai.APITimeoutError, openai.InternalServerError):
            if attempt == RETRIES - 1:
                raise
            time.sleep(2 ** attempt * 5)


def save_settings():
    settings = {
        "case_file": str(CASE_FILE.relative_to(ROOT)), "model": MODEL, "api": "chat.completions",
        "reasoning_effort": "not set (model default)", "system_prompt": None, "tools": None,
        "with_images": WITH_IMAGES, "image_detail": IMAGE_DETAIL if WITH_IMAGES else None,
        "message_order": "prompt text, then picture" if WITH_IMAGES else "prompt text",
        "openai_sdk": openai.__version__, "updated_utc": datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S"),
    }
    previous = json.loads(settings_path.read_text()) if settings_path.exists() else {}
    changed = {k: previous[k] for k in settings if k in previous and previous[k] != settings[k]
               and k not in ("batch_ids", "pending_batch_ids", "updated_utc", "anthropic_sdk", "openai_sdk")}
    if changed:                                     # answers already in the file were made with these earlier values
        settings["earlier_settings"] = previous.get("earlier_settings", []) + [changed]
    settings_path.write_text(json.dumps({**previous, **settings}, indent=2) + "\n")

## 4. Try one case

Sends the first case and prints the parsed answer next to the truth, and the token counts: `cached` is the part of the prompt read from the cache (0 on a first call), `reasoning` the model's thinking tokens.

In [ ]:
case = cases[0]
reply = run_prompt(case)
u = reply.usage
cached = getattr(u.prompt_tokens_details, "cached_tokens", 0) if u.prompt_tokens_details else 0
reasoning = getattr(u.completion_tokens_details, "reasoning_tokens", 0) if u.completion_tokens_details else 0
print(f"[{reply.model} | prompt: {u.prompt_tokens} (cached {cached}) | output: {u.completion_tokens} (reasoning {reasoning})]")
print(parse(reply.choices[0].message.content or ""))
print("truth:", {k: case[k] for k in ["name", "season", "basin", "ri_tplus24", "usa_wind_tplus24h"] if k in case})

## 5. Run every case

Sends each case not yet in the results file, `WORKERS` at a time, and appends each reply to the working results file (`.jsonl`) as soon as it arrives, so a stopped run loses nothing. At the end it writes the compressed copy (`.jsonl.gz`), which is the file to commit. Cases that still fail after the retries are listed; run this cell again to resend just those.

In [ ]:
done = set()
if results_path.exists():
    done = {json.loads(line)["case_id"] for line in results_path.open()}
todo = [c for c in cases if c["case_id"] not in done]
print(f"{len(done)} cases already answered, {len(todo)} to send")
if todo:
    save_settings()

failed = []
with concurrent.futures.ThreadPoolExecutor(max_workers=WORKERS) as pool, results_path.open("a") as out:
    futures = {pool.submit(run_prompt, c): c["case_id"] for c in todo}
    for future in tqdm(concurrent.futures.as_completed(futures), total=len(futures)):
        case_id = futures[future]
        try:
            reply = future.result()
        except Exception as err:
            failed.append((case_id, f"{type(err).__name__}: {err}"))
            continue
        out.write(json.dumps({"case_id": case_id, "response": reply.model_dump(mode="json")}) + "\n")
        out.flush()

print(f"{len(todo) - len(failed)} answers saved; {len(failed)} failed")
for case_id, err in failed[:10]:
    print("  ", case_id, err[:200])

# Compressed copy of all answers: this is the file to commit (plain .jsonl files are not in git).
# Not for a LIMIT trial: notebooks 08 and 09 expect a results file to answer every case.
if LIMIT:
    print(f"LIMIT run: answers kept only in {results_path.name}; run without LIMIT to write the file to commit")
else:
    with open(results_gz, "wb") as raw, gzip.GzipFile(fileobj=raw, mode="wb", mtime=0) as gz:
        gz.write(results_path.read_bytes())
    print(f"Saved {results_gz}")

## 6. Check the answers

Parses every saved answer with the same parser used in the analysis, and summarises the run: unreadable or cut-off answers, and mean prompt, cached and reasoning tokens.

In [ ]:
from collections import Counter

rows = [json.loads(line) for line in results_path.open()]
finish = Counter(r["response"]["choices"][0]["finish_reason"] for r in rows)
bad = []
for r in rows:
    try:
        parse(response_text(r["response"]))
    except ValueError as err:
        bad.append((r["case_id"], str(err)))
usage = [r["response"]["usage"] for r in rows]
mean = lambda xs: sum(xs) / max(len(xs), 1)
print(f"{len(rows)} of {len(cases)} cases answered | finish reasons: {dict(finish)} | unreadable: {len(bad)}")
print(f"mean tokens: prompt {mean([u['prompt_tokens'] for u in usage]):.0f}, "
      f"cached {mean([(u.get('prompt_tokens_details') or {}).get('cached_tokens') or 0 for u in usage]):.0f}, "
      f"output {mean([u['completion_tokens'] for u in usage]):.0f}, "
      f"reasoning {mean([(u.get('completion_tokens_details') or {}).get('reasoning_tokens') or 0 for u in usage]):.0f}")
for case_id, err in bad[:10]:
    print("  ", case_id, err)

## Notes

- **Same results format as before:** `{case_id, response}` per line, so notebooks 08 and 09 read these files directly.
- **Image token cost:** a combined picture is 1042 × 578 px; at `detail: "high"` it costs roughly 750 image tokens on the GPT-5.4 and GPT-5.6 models (the exact count is in each reply's `prompt_tokens`).
- **Knowledge cutoffs:** GPT-5 mini 31 May 2024; GPT-5.4 mini 31 Aug 2025; GPT-5.6 Luna 16 Feb 2026; GPT-6 Luna 18 May 2026. Test storms after the cutoff: 16 for GPT-5 mini, 2 for GPT-5.4 mini (Imelda and Octave, 2025), none for the Luna models.
- **Never send file names:** pictures go as bytes inside the request; the `case_id` stays on our side.